In [14]:
import mlflow
from torchinfo import summary
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
import lightning as L
from mlflow_demo.xai_eval import rcap
from mlflow_demo.xai import gradient_methods
from tqdm import tqdm

L.seed_everything(42)
mlflow.set_tracking_uri('http://localhost:28080')
device = "mps"
# device = "cuda"
batch_size = 32

Seed set to 42


### Dataset


In [12]:
user_home = os.path.expanduser('~')
data_root = os.path.join(user_home, 'autodl-tmp',
                         'ml_data', 'cifar', 'cifar10', 'cifar10')

trainsform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root=os.path.join(data_root, 'train'), transform=trainsform)
test_dataset = datasets.ImageFolder(
    root=os.path.join(data_root, 'test'), transform=trainsform)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size,
                             shuffle=False, num_workers=0)

output_features = 10

### Model


In [13]:
from torchvision import models
import torchmetrics

mlflow.pytorch.autolog()


class MyLightModel(L.LightningModule):
    def __init__(
        self,
        output_features,
        loss_fn_key="CrossEntropyLoss",
        loss_fn_hparams: dict = {},
        optimizer_key="Adam",
        optimizer_hparams: dict = {},
        task="multiclass",
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    ):
        super().__init__()
        self.save_hyperparameters(ignore=[])

        model = models.efficientnet_v2_s(
            weights=weights
        )
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(
            in_features=in_features, out_features=output_features
        )
        self.model = model

        self.loss_fn = getattr(torch.nn, loss_fn_key)(**loss_fn_hparams)

        self.metrics = dict(
            acc=torchmetrics.Accuracy(
                task=task, top_k=1, num_classes=output_features),
            macro_acc=torchmetrics.Accuracy(
                task=task, top_k=1, average="macro", num_classes=output_features
            ),
            f1=torchmetrics.F1Score(
                task=task, average="macro", num_classes=output_features),
            roc_auc=torchmetrics.AUROC(task=task, num_classes=output_features),
        )

        self.st = None
        self.y_true = np.array([])
        self.y_pred = np.array([])
        self.losses = np.array([])
        self.cm = None
        self.cr = None
        self.metrics_in_batch = {}

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        output = self.model(x)
        loss = self.loss_fn.to(x.device)(output, y)
        self.eval_metrics("train", batch, batch_idx, output, loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        output = self.model(x)
        loss = self.loss_fn.to(x.device)(output, y)
        self.eval_metrics("val", batch, batch_idx, output, loss)

    def configure_optimizers(self):
        params = self.model.parameters()

        self.optimizer = getattr(torch.optim, self.hparams.optimizer_key)(
            params, **self.hparams.optimizer_hparams
        )
        return self.optimizer

    def eval_metrics(self, event_key, batch, batch_idx, output, loss):
        x, y = batch
        metrics = {}
        if self.hparams.task == "multiclass":
            _y = y.detach().cpu()
            odc = output.detach().cpu()
            for k, v in self.metrics.items():
                key = f"{event_key}_{k}"

                # we have to make the y to cpu to avoid MPS glitch
                if len(_y.shape) > 1:
                    _y = _y.argmax(dim=1)
                score = v(odc.float(), _y).item()
                metrics[key] = score
                # self.trainer.progress_bar_metrics[k] = metrics[key]
                if event_key == "val":
                    if self.metrics_in_batch.get(key) is None:
                        self.metrics_in_batch[key] = []
                    self.metrics_in_batch[key].append(score)

            if event_key == "val":
                self.y_true = np.append(self.y_true, _y.numpy())
                self.y_pred = np.append(
                    self.y_pred, torch.argmax(odc, dim=1).numpy()
                )
                self.losses = np.append(self.losses, loss.cpu().item())

                ytp = torch.from_numpy(self.y_pred)
                ytt = torch.from_numpy(self.y_true)
                overall_val_acc = self.metrics["acc"](ytp, ytt)
                self.trainer.progress_bar_metrics["max_val_acc"] = max(
                    overall_val_acc,
                    self.trainer.progress_bar_metrics.get("max_val_acc", 0.0),
                )

        metrics[f"{event_key}_loss"] = loss.item()
        self.trainer.progress_bar_metrics["loss"] = metrics[f"{event_key}_loss"]
        self.log_dict(metrics)


model = MyLightModel(
    10,
    optimizer_hparams=dict(
        lr=0.0001,
        weight_decay=4.0e-05
    )
)
model.hparams

"loss_fn_hparams":   {}
"loss_fn_key":       CrossEntropyLoss
"optimizer_hparams": {'lr': 0.0001, 'weight_decay': 4e-05}
"optimizer_key":     Adam
"output_features":   10
"task":              multiclass
"weights":           EfficientNet_V2_S_Weights.IMAGENET1K_V1

### MLflow


In [16]:
trainer_params = dict(
    max_epochs=1,
    limit_train_batches=1,
    limit_test_batches=1,
    limit_val_batches=1,
    callbacks=[
        L.pytorch.callbacks.ModelCheckpoint(
            monitor='val_acc',
            save_top_k=1,
            filename='best-{epoch}-{val_acc:.4f}',
            mode='max',
            save_weights_only=True
        )
    ],

)
trainer = L.Trainer(
    **trainer_params
)

with mlflow.start_run():
    run = mlflow.active_run()
    # Log training parameters.
    mlflow.log_params(model.hparams)
    mlflow.log_params({'trainer_params': trainer_params})

    # Log model summary.
    artifact_root_path = os.path.join("runs", run.info.run_id)
    model_summary_path = os.path.join(artifact_root_path, "model_summary.txt")
    os.makedirs(os.path.dirname(model_summary_path), exist_ok=True)
    with open(model_summary_path, "w") as f:
        f.write(str(summary(model)))
    mlflow.log_artifact(model_summary_path)

    print("Start train")
    trainer.fit(model, train_dataloader)

    print("Start val")
    trainer.validate(model, train_dataloader)

    print("Skip XAI")

    print("XAI Eval")
    model.eval()
    all = []
    for images, targets in tqdm(test_dataloader):
        images, targets = images.to(device), targets.to(device)
        rs = rcap.batch_rcap(
            model.to(device), (images, targets),
            gradient_methods.guided_absolute_grad,
            {}
        )['overall_rcap']['RCAP']
        mlflow.log_metric('rcap', rs.mean())
        all.extend(rs)
        break
    mlflow.log_metric('rcap', np.array(all).mean())

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (mps), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.

  | Name    | Type             | Params
---------------------------------------------
0 | model   | EfficientNet     | 20.2 M
1 | loss_fn | CrossEntropyLoss | 0     
---------------------------------------------
20.2 M    Trainable params
0         Non-trainable params
20.2 M    Total params
80.761    Total estimated mo

Start train


Training: |          | 0/? [00:00<?, ?it/s]

2025/07/27 22:46:06 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not available.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.
2025/07/27 22:46:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Start val


...

Skip XAI
XAI Eval


  0%|          | 0/313 [00:00<?, ?it/s]

Get saliency maps


  0%|          | 0/313 [00:01<?, ?it/s]

Do get_visulalization_and_localization_score
Do get_rcap_score
🏃 View run burly-bug-1 at: http://localhost:28080/#/experiments/0/runs/0642a24692e1435d8a4becf0fd1e794a
🧪 View experiment at: http://localhost:28080/#/experiments/0
